# Ejemplos Nueva Arquitectura (Actualizado)

Flujo completo del Portfolio Tracker con la arquitectura multi-proveedor:

| Sección | Descripción |
|---|---|
| **1. Inicializar Portfolio** | Carga Excel de Órdenes + FIFO automático |
| **2. Posiciones Activas** | Valoración en tiempo real (FMP → YFinance → MStar) |
| **3. Enriquecimiento de datos** | Todos los fondos × todos los proveedores |
| **4. Calculadora Fiscal** | Retirada optimizada con mínimo impacto fiscal |

**Proveedores de datos enriquecidos:**
- 🏦 **Finect** — Comisiones, info general, categoría, gestora (TODOS los fondos UCITS)
- 📰 **Financial Times** — Sectores, Regiones, Asset Allocation, Holdings (scraping FT)
- ⭐ **Morningstar** — Rating (estrellas), categoría, riesgo (via mstarpy)
- 📈 **FMP / YFinance** — Fallback para ETFs y fondos no cubiertos

---
## 1. Inicializar Portfolio desde Excel

In [ ]:
import sys, os, warnings
import pandas as pd
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# Resolver rutas según desde dónde se lance el notebook
if os.path.exists('../app'):
    sys.path.append(os.path.abspath('..'))
    path = '../data/Ordenes.xlsx'
else:
    sys.path.append(os.path.abspath('./backend'))
    path = './backend/data/Ordenes.xlsx'

from app.services.core_portfolio import Portfolio

portfolio = Portfolio(path)
isins_cartera = list(portfolio.positions.keys())

print(f"✅ Portfolio inicializado con éxito.")
print(f"   Lotes abiertos: {len(portfolio.get_open_lots())}")
print(f"   Fondos en cartera: {len(isins_cartera)}")
print(f"   ISINs: {isins_cartera}")

---
## 2. Posiciones Activas

NAV obtenido vía cadena rápida: **FMP → YFinance** (sin esperas de scraping).

In [ ]:
print("Obteniendo valoraciones...")
df_posiciones = portfolio.get_positions()
display(df_posiciones)

print(f"\n  Fondos en cartera:  {len(df_posiciones)}")
print(f"  Capital Invertido:  {portfolio.get_total_invested():,.2f} €")

---
## 3. Enriquecimiento de Datos Financieros

### Setup de proveedores

Instanciamos cada proveedor **independientemente** para poder llamar a cada uno de forma explícita y fusionar resultados en el notebook.

In [ ]:
Rehaz el notebook ejemplos_arquitectura para que refleje la reestructuracion de las peticiones de informacion de los fondos. En la parte de enriquecer datos financieros tienen que aparecer lo siguiente: 
1. Comisiones
2. Sectores
3. Regiones
4. Asset Allocation
5. Portfolio
6. Estrellas Morningstar

SyntaxError: invalid syntax (4226041991.py, line 1)

In [ ]:
from app.services.data_providers import CompositeProvider, FMPProvider, MStarProvider
from app.services.ft_provider import FTProvider
from app.services.finect_provider import FinectProvider

composite  = CompositeProvider()   # cadena rápida NAV + FT/YF/FMP para datos
ft         = FTProvider()           # Financial Times — sectores, regiones, holdings, comisiones
finect     = FinectProvider()       # Finect — info, comisiones, categoría (todos los UCITS)
mstar      = MStarProvider()        # Morningstar — rating estrellas, categoría, riesgo
fmp        = FMPProvider()          # FinancialModelingPrep — ETFs, expense_ratio

print("✅ Proveedores inicializados:")
print(f"   - CompositeProvider (FT + YFinance + FMP)")
print(f"   - FTProvider (Financial Times)")
print(f"   - FinectProvider (Finect sitemap)")
print(f"   - MStarProvider  (mstarpy / Morningstar)")
print(f"   - FMPProvider    (disponible: {fmp.available})")

### Recolección de datos — todos los fondos

Un único loop sobre **todos los ISINs** que llama a cada proveedor.  
Los resultados se almacenan en un dict central `datos` para ser consultados en cada sección.

In [ ]:
from typing import Dict, Any

datos: Dict[str, Dict[str, Any]] = {}

n = len(isins_cartera)
for i, isin in enumerate(isins_cartera, 1):
    print(f"[{i:02d}/{n}] {isin} ... ", end="", flush=True)
    
    entry: Dict[str, Any] = {"isin": isin}
    
    # ── Info fusionada: Finect + FT + FMP (primer valor no vacío gana) ──
    info_merged: Dict[str, Any] = {}
    for src_name, src in [("finect", finect), ("ft", ft), ("fmp", fmp)]:
        try:
            info = src.get_fund_info(isin) or {}
            for k, v in info.items():
                if k not in info_merged or not info_merged[k] or info_merged[k] == "—":
                    if v is not None and v != "" and v != isin:
                        info_merged[k] = v
        except Exception:
            pass
    entry["info"] = info_merged
    
    # ── Morningstar: rating (estrellas) + categoría ──
    try:
        ms_info = mstar.get_fund_info(isin) or {}
        entry["morningstar"] = ms_info
        # Fusionar nombre si no lo tenemos
        if "name" not in info_merged and ms_info.get("name"):
            info_merged["name"] = ms_info["name"]
    except Exception:
        entry["morningstar"] = {}
    
    # ── Sectores ──
    try:
        entry["sectors"] = composite.get_sector_weights(isin) or {}
    except Exception:
        entry["sectors"] = {}
    
    # ── Regiones ──
    try:
        entry["regions"] = composite.get_country_weights(isin) or {}
    except Exception:
        entry["regions"] = {}
    
    # ── Asset Allocation (FT primero, luego composite) ──
    try:
        alloc = ft.get_asset_allocation(isin)
        if not alloc:
            alloc = composite.get_asset_allocation(isin) or {}
        entry["asset_allocation"] = alloc
    except Exception:
        entry["asset_allocation"] = {}
    
    # ── Holdings / Portfolio (FT primero, luego composite) ──
    try:
        holdings = ft.get_holdings(isin)
        if holdings is None or holdings.empty:
            holdings = composite.get_holdings(isin)
        entry["holdings"] = holdings
    except Exception:
        entry["holdings"] = None
    
    datos[isin] = entry
    
    # Resumen de cobertura para este fondo
    cov = []
    if info_merged.get("name"): cov.append("info")
    if entry["morningstar"].get("overallMorningstarRating"): cov.append("⭐")
    if entry["sectors"]: cov.append("sectores")
    if entry["regions"]: cov.append("regiones")
    if entry["asset_allocation"]: cov.append("allocation")
    if entry["holdings"] is not None and not entry["holdings"].empty: cov.append("holdings")
    print(", ".join(cov) if cov else "sin datos")

print(f"\n✅ Datos recolectados para {len(datos)} fondos.")

---
### 3.1 Comisiones + Rating Morningstar

Tabla resumen con **todos los fondos**: nombre, comisiones y estrellas Morningstar.  
Fuentes: **Finect** (comisiones UCITS), **FT** (ongoing charge ETFs), **FMP** (expense_ratio), **Morningstar** (rating).

In [ ]:
def stars(rating):
    """Convierte rating numérico de Morningstar en ⭐."""
    try:
        n = int(float(rating))
        return "⭐" * n if 1 <= n <= 5 else str(rating)
    except (TypeError, ValueError):
        return "—"

rows = []
for isin, entry in datos.items():
    info = entry["info"]
    ms   = entry["morningstar"]
    
    # Nombre — Finect > FT > Morningstar > ISIN
    nombre = info.get("name") or ms.get("name") or isin
    
    # Comisiones — intentar varios campos
    gasto_corriente = (
        info.get("ongoing_charge")            # FT / Finect
        or info.get("comision_de_gestion")    # Finect
        or ms.get("ongoingCostsOtherCosts")   # Morningstar
        or info.get("expense_ratio")           # FMP
        or "—"
    )
    comision_inicial = info.get("initial_charge") or info.get("comision_de_suscripcion") or "—"
    categoria = info.get("category") or ms.get("categoryName") or "—"
    
    rows.append({
        "ISIN": isin,
        "Nombre": nombre[:55] + ("..." if len(str(nombre)) > 55 else ""),
        "Categoría": str(categoria)[:40],
        "Gasto corriente": gasto_corriente,
        "C. Suscripción": comision_inicial,
        "⭐ Morningstar": stars(ms.get("overallMorningstarRating")),
        "Fuente info": info.get("source", "—"),
    })

df_comisiones = pd.DataFrame(rows)
print(f"=== 3.1 COMISIONES + RATING MORNINGSTAR ({len(df_comisiones)} fondos) ===")
display(df_comisiones.set_index("ISIN"))

### 3.2 Sectores

Distribución sectorial de cada fondo. Fuente: **FT → YFinance → FMP** (la más completa gana).  
Los fondos sin datos de sectores (ej. fondos de renta fija puros) aparecen como `—`.

In [ ]:
print("=== 3.2 SECTORES ===")

# Construir tabla pivot: filas=ISIN, columnas=sectores únicos
all_sectors = set()
for entry in datos.values():
    all_sectors.update(entry["sectors"].keys())

if all_sectors:
    rows_sec = []
    for isin, entry in datos.items():
        nombre = (entry["info"].get("name") or entry["morningstar"].get("name") or isin)[:40]
        row = {"ISIN": isin, "Fondo": nombre}
        for s in sorted(all_sectors):
            val = entry["sectors"].get(s)
            row[s] = f"{val:.1f}%" if val is not None else "—"
        rows_sec.append(row)
    df_sec = pd.DataFrame(rows_sec).set_index("ISIN")
    display(df_sec)
else:
    print("  ⚠️  No se obtuvieron datos de sectores para ningún fondo.")

# Detalle individual para los que sí tienen datos
print("\n--- Detalle por fondo ---")
for isin, entry in datos.items():
    sectores = entry["sectors"]
    if sectores:
        nombre = (entry["info"].get("name") or isin)[:50]
        print(f"\n📌 {isin} — {nombre}")
        df_s = pd.DataFrame(list(sectores.items()), columns=["Sector", "Peso (%)"])
        df_s = df_s.sort_values("Peso (%)", ascending=False).reset_index(drop=True)
        display(df_s)

### 3.3 Regiones

Distribución geográfica por región/país. Fuente: **FT → YFinance → FMP**.

In [ ]:
print("=== 3.3 REGIONES ===")

for isin, entry in datos.items():
    regiones = entry["regions"]
    nombre = (entry["info"].get("name") or entry["morningstar"].get("name") or isin)[:55]
    if regiones:
        print(f"\n🌍 {isin} — {nombre}")
        df_r = pd.DataFrame(list(regiones.items()), columns=["Región / País", "Peso (%)"])
        df_r = df_r.sort_values("Peso (%)", ascending=False).reset_index(drop=True)
        display(df_r)
    else:
        print(f"\n🌍 {isin} — {nombre}: ⚠️ sin datos de regiones")

### 3.4 Asset Allocation

Desglose por clase de activo (Renta Variable, Renta Fija, Liquidez, etc.).  
Fuente: **Financial Times** (holdings page) → CompositeProvider como fallback.

In [ ]:
print("=== 3.4 ASSET ALLOCATION ===")

# Tabla resumen (solo fondos con datos)
all_asset_classes = set()
for entry in datos.values():
    all_asset_classes.update(entry["asset_allocation"].keys())

rows_alloc = []
for isin, entry in datos.items():
    alloc = entry["asset_allocation"]
    nombre = (entry["info"].get("name") or entry["morningstar"].get("name") or isin)[:45]
    if alloc:
        row = {"ISIN": isin, "Fondo": nombre}
        for ac in sorted(all_asset_classes):
            val = alloc.get(ac)
            row[ac] = f"{val:.1f}%" if val is not None else "—"
        rows_alloc.append(row)

if rows_alloc:
    df_alloc = pd.DataFrame(rows_alloc).set_index("ISIN")
    print(f"\nFondos con Asset Allocation disponible: {len(rows_alloc)} / {len(datos)}")
    display(df_alloc)
else:
    print("  ⚠️  No se obtuvieron datos de asset allocation.")

# Listar los que no tienen datos
sin_datos = [isin for isin, e in datos.items() if not e["asset_allocation"]]
if sin_datos:
    print(f"\nSin datos de asset allocation: {sin_datos}")

### 3.5 Portfolio (Top Holdings)

Principales posiciones en cartera de cada fondo.  
Fuente: **Financial Times** (holdings tearsheet) → YFinance → FMP.

In [ ]:
print("=== 3.5 PORTFOLIO — TOP HOLDINGS ===")

for isin, entry in datos.items():
    holdings = entry["holdings"]
    nombre = (entry["info"].get("name") or entry["morningstar"].get("name") or isin)[:55]
    print(f"\n🏦 {isin} — {nombre}")
    if holdings is not None and not holdings.empty:
        # Mostrar top 10, ordenadas por peso descendente
        h = holdings.copy()
        if "weight" in h.columns:
            h = h.sort_values("weight", ascending=False)
        display(h.head(10).reset_index(drop=True))
    else:
        print("   ⚠️  Sin datos de holdings disponibles.")

### 3.6 Resumen de cobertura por proveedor

Mapa de qué datos se obtuvieron para cada fondo.

In [ ]:
print("=== RESUMEN DE COBERTURA ===")

rows_cov = []
for isin, entry in datos.items():
    info = entry["info"]
    ms   = entry["morningstar"]
    nombre = (info.get("name") or ms.get("name") or isin)[:45]
    rows_cov.append({
        "ISIN": isin,
        "Nombre": nombre,
        "Info/Comisiones": "✅" if info.get("name") else "❌",
        "⭐ MStar": stars(ms.get("overallMorningstarRating")),
        "Sectores": "✅" if entry["sectors"] else "❌",
        "Regiones": "✅" if entry["regions"] else "❌",
        "Asset Alloc": "✅" if entry["asset_allocation"] else "❌",
        "Holdings": "✅" if (entry["holdings"] is not None and not entry["holdings"].empty) else "❌",
    })

df_cov = pd.DataFrame(rows_cov).set_index("ISIN")
display(df_cov)

---
## 4. Calculadora Fiscal — Retirada Optimizada

El `TaxOptimizer` analiza los lotes FIFO y sugiere qué vender para minimizar la ganancia patrimonial aflorada, según la legislación española.

In [ ]:
from app.services.tax_calculator import TaxOptimizer

cantidad_a_retirar = 50000
print(f"Simulando una retirada optimizada de {cantidad_a_retirar:,}€...")

optimizer = TaxOptimizer(portfolio)
plan = optimizer.optimize_withdrawal(cantidad_a_retirar)

print("\n=== Plan de Retirada ===")
print(f"  Cantidad Bruta Retirada:          {plan['withdrawn_amount']:>12,.2f} €")
print(f"  Ganancia Patrimonial Aflorada:    {plan['total_capital_gain']:>12,.2f} €")
print(f"  Impuestos Estimados a Pagar:      {plan['estimated_tax']:>12,.2f} €")
print(f"  Cantidad Neta en tu cuenta:       {plan['net_amount']:>12,.2f} €")

print("\n--- Ventas recomendadas (lotes específicos) ---")
rows_plan = []
for step in plan['plan']:
    rows_plan.append({
        "Fondo (ISIN)": step['Fondo'],
        "Fecha Compra": step['Fecha_Compra'].strftime('%Y-%m-%d'),
        "Participaciones": f"{step['Participaciones_Vendidas']:.4f}",
        "Importe Retirado": f"{step['Importe_Retirado']:,.2f} €",
    })
display(pd.DataFrame(rows_plan))